# Classify Tree Species at CLBJ with Random Forest in the Earth Engine Python API

This tutorial classifies woody plant species and three land cover types (grassland, water, and shade) at the NEON Lyndon B. Johnson National Grassland (CLBJ, domain D11) site. It uses 2017 NEON Airborne Observation Platform (AOP) hyperspectral reflectance and lidar canopy height data, NEON vegetation structure data as reference data, and a random forest classifier that runs in Google Earth Engine (GEE).

This notebook is a port of a GEE Code Editor (JavaScript) script to the GEE Python API. In this first version the Python code reproduces the JavaScript workflow step by step, so that the accuracy numbers match the JavaScript output. Some choices in the original method are not ideal for a robust classification. They are kept on purpose, and they are marked in the code comments. They are listed at the end of the notebook.

Adapted from Geethen Singh, 2020, GEE for Ecology and Conservation OTS course.

## Background Tutorials

This tutorial builds on methods and tools from the following NEON tutorials:

* [Using neonDataStackR to access NEON data](https://www.neonscience.org/resources/learning-hub/tutorials/neondatastackr)
* [Work with NEON AOP reflectance data in Python with geemap](https://www.neonscience.org/resources/learning-hub/tutorials/aop-refl-py-geemap)
* [Classify tree species with NEON AOP data and random forest classification in Google Earth Engine](https://www.neonscience.org/resources/learning-hub/tutorials/aop-gee-random-forest-classification)

## Learning Objectives

## Data Products and Study Site

The notebook uses the following NEON data products. Earth Engine hosts the AOP products as image collections and the vegetation reference points as a feature collection.

| NEON data product | ID | Earth Engine asset |
|---|---|---|
| Spectrometer orthorectified surface directional reflectance - mosaic | [DP3.30006.001](https://data.neonscience.org/data-products/DP3.30006.001) | `projects/neon/DP3-30006-001_SDR` |
| Ecosystem structure (canopy height model, CHM) | [DP3.30015.001](https://data.neonscience.org/data-products/DP3.30015.001) | `projects/neon/DP3-30024-001_DEM` (filter `Type` = `CHM`) |
| Elevation - LiDAR (digital terrain model, DTM) | [DP3.30024.001](https://data.neonscience.org/data-products/DP3.30024.001) | `projects/neon/DP3-30024-001_DEM` (filter `Type` = `DTM`) |
| Woody plant vegetation structure | [DP1.10098.001](https://data.neonscience.org/data-products/DP1.10098.001) | `projects/neon/AOP_NEON_Plots/CLBJ_veg_2017_filtered` |

In the Earth Engine asset used here, the CHM and the DTM are stored in the same image collection (`DP3-30024-001_DEM`), and you tell them apart with the `Type` property.

The reference points are stem locations of woody plants, with the NEON `taxonID` of each stem. Points for three land cover classes (grassland `GRSS`, water `WATR`, and shade `SHADE`) were added later by visual inspection of the reflectance data. Two more assets give the study area: the NEON Terrestrial Observation System (TOS) boundary (`projects/neon/AOP_TOS_Boundaries/D11_CLBJ_TOS_dissolve`) and the 90 percent footprint of the tower airshed (`projects/neon/Airsheds/CLBJ_90percent_footprint`).

**Site:** CLBJ, Lyndon B. Johnson National Grassland, NEON domain D11 (Texas). **Year:** 2017.

## Setup

Import the packages and initialize Earth Engine. Earth Engine requires a Google Cloud project for every session.

In [ ]:
import functools

import ee
import geemap
import numpy as np
import pandas as pd

# Change this to the ID of your own Google Cloud project that is registered for Earth Engine.
PROJECT_ID = 'neon-sandbox-dataflow-ee'

ee.Initialize(project=PROJECT_ID)

## Workflow

1. Specify the input (training) data for display and use in the analysis
2. Combine the AOP spectrometer reflectance data with the lidar-derived canopy height model (CHM) to create the predictor data
3. Create reference data for plant species and land cover types, based on the NEON vegetation structure data
4. Fit a random forest model, based on the spectral/CHM composite, to map the distribution of plant species at CLBJ
5. Evaluate the model accuracy

## Step 1: Specify the Input Data

First, specify the CLBJ location as a point and load the three vector datasets: the vegetation reference points, the TOS boundary, and the tower airshed boundary. These are `FeatureCollection` objects. Earth Engine does not download them. It only stores a reference to the server-side data.

In [ ]:
# Specify the CLBJ location (longitude, latitude)
geo = ee.Geometry.Point([-97.5706464, 33.4045729])

# Species and land cover samples. Originally a .csv file extracted from the NEON
# woody plant vegetation structure data product (DP1.10098.001)
CLBJ_veg = ee.FeatureCollection('projects/neon/AOP_NEON_Plots/CLBJ_veg_2017_filtered')

# Terrestrial Observation System (TOS) boundary
CLBJ_TOS = ee.FeatureCollection('projects/neon/AOP_TOS_Boundaries/D11_CLBJ_TOS_dissolve')

# Tower airshed boundary
CLBJ_Airshed = ee.FeatureCollection('projects/neon/Airsheds/CLBJ_90percent_footprint')

### Digital terrain model and canopy height model

The DTM and the CHM are in the same image collection. Filter the collection by year, by the `Type` property, and by location, and take the first image that remains. The raster files use -9999 as the no-data value, so the display versions of the images are masked to values greater than or equal to 0.

In [ ]:
dem_collection = ee.ImageCollection('projects/neon/DP3-30024-001_DEM')


def get_dem_2017(dem_type):
    """First 2017 image of the given Type ('DTM' or 'CHM') that intersects the CLBJ point."""
    return (dem_collection
            .filterDate('2017-01-01', '2017-12-31')
            .filter(ee.Filter.eq('Type', dem_type))
            .filterBounds(geo)
            .first())


# Digital Terrain Model (DTM): mask the no-data values (-9999) for display
CLBJ_DTM2017 = get_dem_2017('DTM')
CLBJ_DTM2017mask = CLBJ_DTM2017.updateMask(CLBJ_DTM2017.gte(0.0000))

# Canopy Height Model (CHM): mask the no-data values (-9999). The masked CHM is also a predictor later.
CLBJ_CHM2017 = get_dem_2017('CHM')
CLBJ_CHM2017mask = CLBJ_CHM2017.updateMask(CLBJ_CHM2017.gte(0.0000))

### Surface directional reflectance (SDR)

The reflectance image has 426 bands. Water vapor absorbs most of the light in two wavelength ranges (band195-band205 and band287-band310), so those bands contain mostly noise. The last 10 bands (band416-band425) are also noisy. Remove these 45 bands, which leaves 381 bands for the classification.

In [ ]:
CLBJ_SDR2017 = (ee.ImageCollection('projects/neon/DP3-30006-001_SDR')
                .filterDate('2017-01-01', '2017-12-31')
                .filterBounds(geo)
                .first())

bandNames = CLBJ_SDR2017.bandNames()

# Water absorption bands (band195-band205, band287-band310) and the last noisy bands (band416-band425)
bandsToRemove = [f'band{i:03d}' for i in [*range(195, 206), *range(287, 311), *range(416, 426)]]
bandsToKeep = bandNames.removeAll(bandsToRemove)
CLBJ_SDR2017subset = CLBJ_SDR2017.select(bandsToKeep)

print('Bands removed:', len(bandsToRemove))
print('CLBJ_SDR2017 valid band subset, number of bands:', bandsToKeep.size().getInfo())  # 381 expected

# Display version: mask the no-data values (-9999) and keep three bands for a red-green-blue (RGB) display
CLBJ_SDR2017mask = (CLBJ_SDR2017.updateMask(CLBJ_SDR2017.gte(0.0000))
                    .select(['band053', 'band035', 'band019']))

## Step 2: Create the Predictor Image

Combine the 381 reflectance bands and the one CHM band into one multiband image, and clip it to the airshed boundary. The classifier uses all bands of this image as predictors.

In [ ]:
# Combine the SDR (381 bands) and the CHM (1 band) for classification.
# Kept as in the JavaScript for parity; see the Corrections section.
# (The reflectance bands are NOT masked for no-data here. Only the CHM is masked.)
CLBJ_SDR_CHMcomposite = CLBJ_SDR2017subset.addBands(CLBJ_CHM2017mask)

# Crop the composite to the NEON tower airshed boundary.
# To classify the entire SDR scene instead, use CLBJ_SDR_CHMcomposite directly.
CLBJ_SDR2017_airshed = CLBJ_SDR_CHMcomposite.clip(CLBJ_Airshed)

print('Composite SDR and CHM image, number of bands:', CLBJ_SDR2017_airshed.bandNames().size().getInfo())

### Map 1: input data

The next cell displays the input layers on an interactive map. Use the layer control (top right of the map) to switch layers on and off. As in the Code Editor script, only the airshed composite is visible at first. The other layers are off until you switch them on. The JavaScript script puts all layers on one map. This notebook splits them over three maps, so that each map follows its workflow step.

In [ ]:
map_inputs = geemap.Map()

map_inputs.addLayer(CLBJ_DTM2017mask, {'min': 285, 'max': 1294}, 'CLBJ DTM 2017', False)
map_inputs.addLayer(CLBJ_CHM2017mask, {'min': 0, 'max': 33}, 'CLBJ CHM 2017', False)
map_inputs.addLayer(CLBJ_SDR2017mask, {'min': 0.5, 'max': 10}, 'CLBJ SDR 2017', False)
map_inputs.addLayer(CLBJ_SDR2017_airshed,
                    {'bands': ['band053', 'band035', 'band019'], 'min': 0.5, 'max': 10},
                    'CLBJ-Airshed SDR/CHM 2017', True)

# Outlines with a transparent fill
map_inputs.addLayer(CLBJ_TOS.style(width=3, color='blue', fillColor='#00000000'), {}, 'CLBJ TOS', False)
map_inputs.addLayer(CLBJ_Airshed.style(width=3, color='white', fillColor='#00000000'), {}, 'CLBJ Airshed', False)

map_inputs.centerObject(geo, 16)
map_inputs

## Step 3: Create the Reference Data

The reference data come from the NEON woody plant vegetation structure data product (DP1.10098.001). The feature collection has one point for each stem, with a `taxonID` (species or land cover code) and a numeric class label `taxonIDnum`. Land cover classes (grassland, water, and shade) were added to the collection after visual inspection of the reflectance data (DP3.30006.001).

The classes, in the order of their `taxonIDnum` label (0 to 9), with the colors that the maps use:

In [ ]:
# (taxonID, common name, map color). The position in this list is the taxonIDnum label (0-9).
CLASSES = [
    ('CELA', 'Sugarberry', 'yellow'),
    ('JUVI', 'Eastern red cedar', 'white'),
    ('PRME', 'Mexican plum', 'green'),
    ('QUMA3', 'Blackjack oak', 'orange'),
    ('QUST', 'Post oak', 'darkgreen'),
    ('ULAL', 'Winged elm', 'cyan'),
    ('ULCR', 'Cedar elm', 'purple'),
    ('GRSS', 'Grassland', 'lightgreen'),
    ('WATR', 'Water', 'blue'),
    ('SHADE', 'Shade', 'black'),
]
CLASS_CODES = [code for code, _, _ in CLASSES]

print('All woody plant samples:', CLBJ_veg.size().getInfo())

# Optional tree height filter, as in the JavaScript (not applied):
# CLBJ_veg_filt = CLBJ_veg.filter(ee.Filter.gt('height', 8))
CLBJ_veg_filt = CLBJ_veg

# One feature collection per class
class_points = {code: CLBJ_veg_filt.filter(ee.Filter.inList('taxonID', [code]))
                for code in CLASS_CODES}

### Map 2: reference points for each class

In [ ]:
map_points = geemap.Map()

# Style each class: add a 'style' property with the marker color and shape to each point
for code, name, color in CLASSES:
    styled_points = class_points[code].map(lambda point: point.set('style', {
        'color': color,
        'pointShape': 'diamond',
        'pointSize': 3,
        'width': 2,
        'fillColor': '00000000',
    }))
    map_points.addLayer(styled_points.style(styleProperty='style'), {}, name, False)

map_points.centerObject(geo, 16)
map_points

### Split the points into training and test sets

Split the points of each class separately (stratified random sampling), so that each class has about 80 percent of its points in the training set and about 20 percent in the test set. For each class, `randomColumn(seed=1)` adds a `random` property with a value between 0 and 1 to each point. Points with `random` below 0.80 are training points. The other points are test points. The seed makes the split repeatable.

Then merge the classes, in the class order above, into one training collection and one test collection. The merge order is kept the same as in the JavaScript script.

In [ ]:
training_parts = []
test_parts = []

for code in CLASS_CODES:
    # Kept as in the JavaScript for parity; see the Corrections section.
    # (Random split of individual stems. Stems from the same plot can be in both sets.)
    with_random = class_points[code].randomColumn(seed=1)
    training_parts.append(with_random.filter(ee.Filter.lt('random', 0.80)))
    test_parts.append(with_random.filter(ee.Filter.gte('random', 0.80)))

# Same as CELAtraining.merge(JUVItraining).merge(PRMEtraining)... in the JavaScript
training = functools.reduce(lambda a, b: a.merge(b), training_parts)
test = functools.reduce(lambda a, b: a.merge(b), test_parts)
points = training.merge(test)

# Number of points in each partition. Small result, so getInfo() is fine.
# Kept as in the JavaScript for parity; see the Corrections section.
# (These counts include points outside the airshed. sampleRegions drops those points in Step 4.)
counts = ee.Dictionary({
    code: {'training': training_parts[i].size(), 'test': test_parts[i].size()}
    for i, code in enumerate(CLASS_CODES)
}).getInfo()

counts_df = pd.DataFrame(counts).T.loc[CLASS_CODES]
counts_df.loc['Total'] = counts_df.sum()
counts_df

## Step 4: Train the Random Forest Classifier

### Extract the predictor values at the training points

`sampleRegions` reads the values of all bands of the composite image at each training point and attaches them to the point as properties. It keeps the class label `taxonIDnum`. The `scale` of 1 meter matches the 1 meter pixel size of the NEON AOP data. `tileScale=16` splits the computation into smaller tiles, which lowers the risk of memory errors.

Each sampled point now has one property for each of the 382 bands, so do not use `getInfo()` on the whole collection. The next cell prints the number of sampled points and the number of properties of the first point only.

In [ ]:
# Kept as in the JavaScript for parity; see the Corrections section.
# (Points outside the airshed polygon are dropped by sampleRegions without a message.)
features = CLBJ_SDR2017_airshed.sampleRegions(
    collection=training,
    properties=['taxonIDnum'],
    scale=1,
    tileScale=16,
)

features_info = ee.Dictionary({
    'n_points': features.size(),
    'n_properties_first_point': features.first().propertyNames().size(),
}).getInfo()
print('Features with spectral signatures:', features_info)

### Train and apply the classifier

Train a random forest with 100 trees (`smileRandomForest`). All other parameters keep the Earth Engine defaults. The input properties are all band names of the composite image. Then classify the whole composite image. The classified image is lazy: Earth Engine computes it only when you display it on a map or export it.

In [ ]:
trained_classifier = ee.Classifier.smileRandomForest(100).train(
    features=features,
    classProperty='taxonIDnum',
    inputProperties=CLBJ_SDR2017_airshed.bandNames(),
)

# Classify the reflectance/CHM image with the trained classifier
classified = CLBJ_SDR2017_airshed.classify(trained_classifier)

# Confusion matrix and overall accuracy for the training sample.
# Kept as in the JavaScript for parity; see the Corrections section.
# (This is a resubstitution accuracy. It overestimates the accuracy because the model has seen these points.)
train_accuracy = trained_classifier.confusionMatrix().accuracy()
print('Train Accuracy', train_accuracy.getInfo())

## Step 5: Evaluate the Model

The test points were not used to train the model, so they give a more reliable accuracy estimate. Sample the composite image at the test points, classify the samples, and compare the predicted class (`classification`) with the reference class (`taxonIDnum`) in an error (confusion) matrix.

In [ ]:
# Extract the spectral signatures at the test points
# Kept as in the JavaScript for parity; see the Corrections section.
# (Test points outside the airshed are also dropped by sampleRegions.)
test_features = CLBJ_SDR2017_airshed.sampleRegions(
    collection=test,
    properties=['taxonIDnum'],
    scale=1,
    tileScale=16,
)

# Classify the test samples and build the error matrix
test_classified = test_features.classify(trained_classifier)
error_matrix = test_classified.errorMatrix('taxonIDnum', 'classification')

# Only small results (one matrix and a few numbers) come back to Python.
metrics = ee.Dictionary({
    'n_test_points': test_features.size(),
    'matrix': error_matrix.array(),
    'overall_accuracy': error_matrix.accuracy(),
    # Kappa coefficient: how well the classification performs compared to random assignment
    'kappa': error_matrix.kappa(),
    # Producer's accuracy: how often a real feature on the ground is correctly shown on the map
    # (the complement of the error of omission)
    'producers_accuracy': error_matrix.producersAccuracy(),
    # User's (consumer's) accuracy: how often a class on the map is really that class on the ground
    # (the complement of the error of commission)
    'users_accuracy': error_matrix.consumersAccuracy(),
}).getInfo()

print('Test points used (inside the airshed):', metrics['n_test_points'])
print('Overall Accuracy:', metrics['overall_accuracy'])
print('Kappa Coefficient:', metrics['kappa'])

### Confusion matrix and per-class accuracy

In the confusion matrix, rows are the reference classes (from the vegetation structure data) and columns are the predicted classes. The matrix has as many rows and columns as the highest class label that occurs in the test data plus one, so a class with no test points can add an empty row. Producer's and user's accuracy are `NaN` for a class that has no reference points (producer's accuracy) or no predicted points (user's accuracy).

In [ ]:
matrix = np.array(metrics['matrix'], dtype=float)
labels = CLASS_CODES[:matrix.shape[0]]

confusion_df = pd.DataFrame(matrix.astype(int), index=labels, columns=labels)
confusion_df.index.name = 'Reference'
confusion_df.columns.name = 'Predicted'
display(confusion_df)

accuracy_df = pd.DataFrame({
    'Producers accuracy': np.array(metrics['producers_accuracy'], dtype=float).ravel(),
    'Users accuracy': np.array(metrics['users_accuracy'], dtype=float).ravel(),
}, index=labels)
display(accuracy_df)

### Map 3: classified image and reference points

The last map shows the classified image and the reference points, each in the colors of the class table above. As in the Code Editor script, the classified image and the points are off at first. Switch them on with the layer control. Earth Engine computes the classified image for the part of the map that you see, so the first display can take a few seconds.

In [ ]:
# Style all reference points (training and test) by class: one diamond marker per point, in the class color
def style_points(class_label, color):
    return (points.filter(ee.Filter.eq('taxonIDnum', class_label))
            .map(lambda point: point.set('style', {
                'color': color,
                'pointShape': 'diamond',
                'pointSize': 3,
                'width': 2,
                'fillColor': '00000000',
            })))


lc_points = functools.reduce(
    lambda a, b: a.merge(b),
    [style_points(i, color) for i, (_, _, color) in enumerate(CLASSES)],
)

palette = [color for _, _, color in CLASSES]

map_results = geemap.Map()

# The composite is repeated here (visible) so that the map has an image under the hidden layers
map_results.addLayer(CLBJ_SDR2017_airshed,
                     {'bands': ['band053', 'band035', 'band019'], 'min': 0.5, 'max': 10},
                     'CLBJ-Airshed SDR/CHM 2017', True)
map_results.addLayer(classified, {'min': 0, 'max': 9, 'palette': palette}, 'Classified image', False)
map_results.addLayer(lc_points.style(styleProperty='style'), {}, 'All training sample points', False)

map_results.centerObject(geo, 16)
map_results

## Next Steps and Known Limitations

This version reproduces the JavaScript method, including its limitations. Take the accuracy numbers above as a starting point, not as a final result. The known problems are:

1. The classification input is not masked for no-data values (-9999). Only the display layers are masked.
2. 26 of the 374 reference points are outside the airshed. `sampleRegions` drops them without a message, so the printed training and test counts are larger than the counts that the model uses.
3. Two classes have too few points for an 80/20 split: `PRME` (3 points) and `ULAL` (6 points). The `PRME` test set has 0 or 1 points.
4. The random split puts stems from the same plot in both the training and the test set, so the test accuracy is probably too high.
5. Stem locations of understory trees are not visible from the air. The `canopyPosition` and `height` properties can be used as filters.
6. "Train Accuracy" is a resubstitution accuracy. The out-of-bag error from `classifier.explain()` is a better estimate.

A Corrections section that addresses these problems will follow in a later version of this tutorial. A later version will also move to the newer NEON reflectance and CHM assets in Earth Engine.